In [3]:
import pandas as pd
import numpy as np

In [4]:
train_df = pd.read_csv("/content/train.csv")
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [5]:
# Frquency Distribution of options as answers.
freq = train_df["answer"].value_counts()
print(freq)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [6]:
max_freq = freq.max()
min_freq = freq.min()
print("Max frequency:", max_freq, "for option:", freq.idxmax())
print("Min frequency:", min_freq, "for option:", freq.idxmin())
print("Sum of max freqency and min frequency is:", max_freq + min_freq)

Max frequency: 490 for option: B
Min frequency: 324 for option: E
Sum of max freqency and min frequency is: 814


2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  


In [7]:
# Convert to lowercase.
train_df["prompt"] = train_df["prompt"].apply(str.lower)
train_df["prompt"].head()

,prompt
0,pick the best possible answer: what is martin ...
1,what is accelerator-based light-ion fusion?
2,determine the correct option: what is the term...
3,select the most accurate option: what is marti...
4,identify the correct statement: what is the co...


In [8]:
# Remove punctuation
import string
def remove_punc(text):
  mod_text = "".join(char for char in text if char not in string.punctuation)
  return mod_text
train_df["prompt"] = train_df["prompt"].apply(remove_punc)
print(train_df["prompt"].head())

0    pick the best possible answer what is martin h...
1             what is acceleratorbased lightion fusion
2    determine the correct option what is the term ...
3    select the most accurate option what is martin...
4    identify the correct statement what is the con...
Name: prompt, dtype: object


In [9]:
def to_words(text):
  words = text.split()
  return words

unique_words = len(train_df["prompt"].apply(to_words).explode().unique())
print("Unqiue words in the training dataset is:", unique_words)

Unqiue words in the training dataset is: 859


3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [10]:
from sklearn import feature_extraction
stop_words = feature_extraction.text.ENGLISH_STOP_WORDS
prompt_row_1 = train_df.loc[train_df["id"] == 1, "prompt"].values[0].split()
print("Prompt in ROW ID 1:", prompt_row_1)
count = 0
for word in prompt_row_1:
  if word not in list(stop_words):
    count += 1
  else:
    print(word)
print("Number of words after removing stop words:", count)

Prompt in ROW ID 1: ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
the
what
is
on
the
between
and
among
the
Number of words after removing stop words: 13


4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    stop_words = "english"
)

In [12]:
# Combine prompt and options
combined = train_df["prompt"] + " " + train_df["A"]  + " " + train_df["B"]  + " " + train_df["C"]  + " " + train_df["D"] + " " + train_df["E"]
print(combined)

0       pick the best possible answer what is martin h...
1       what is acceleratorbased lightion fusion Accel...
2       determine the correct option what is the term ...
3       select the most accurate option what is martin...
4       identify the correct statement what is the con...
                              ...                        
1995    what is the piezoelectric strain coefficient f...
1996    identify the correct statement what is the syn...
1997    determine the correct option what does earnsha...
1998    identify the correct statement what is the rev...
1999    pick the best possible answer which of the fol...
Length: 2000, dtype: object


In [13]:
# Convert to list
combined_list = combined.to_list()
print(combined_list)

["pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. Martin Heidegger believes that the relationship between time and huma

In [14]:
# Apply tfidf
tfidf_vectorizer.fit(combined_list)

TfidfVectorizer(stop_words='english')

In [15]:
print("Vocabulary size generated:", len(tfidf_vectorizer.vocabulary_))

Vocabulary size generated: 2807


5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [16]:
print("Prompt:", train_df.loc[train_df["id"]==1, "prompt"].values[0])
print("Option A:", train_df.loc[train_df["id"]==1, "A"].values[0])
a = train_df.loc[train_df["id"]==1, "prompt"].values[0]
b = train_df.loc[train_df["id"]==1, "A"].values[0]
document = [a,b]

Prompt: pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
Option A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


In [17]:
from sklearn.metrics.pairwise import cosine_similarity
vectors = tfidf_vectorizer.transform(document)
similarity = cosine_similarity(vectors, vectors)
print(similarity)

[[1.         0.18475412]
 [0.18475412 1.        ]]


In [18]:
print(f"Similarity between Prompt and Option A is:{similarity[0][1]:.4f}")

Similarity between Prompt and Option A is:0.1848


6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [19]:
correct_predictions = 0
total_questions = len(train_df)
options = ["A","B","C","D","E"]


for index, row in train_df.iterrows():
  prompt = row['prompt']
  options_data = [row["A"],row["B"],row["C"], row["D"],row["E"]]

  document = [prompt] + options_data

  vectors = tfidf_vectorizer.transform(document)

  similarities = cosine_similarity(vectors[:1],vectors[1:]).flatten()

  best_index = np.argmax(similarities)

  predicted_answer = options[best_index]

  if predicted_answer == row['answer']:
    correct_predictions += 1

accuracy = (correct_predictions/total_questions) * 100
print(accuracy)

print("Number of Correct Predictions: ", correct_predictions)
print("The percentage of instances where the option with the highest cosine" \
"similarity matches the correct answer.", accuracy)

13.05
Number of Correct Predictions:  261
The percentage of instances where the option with the highest cosinesimilarity matches the correct answer. 13.05


7. If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

In [20]:
def calculate_ap3(truth, prediction):
  predictions = prediction.split()

  if truth == predictions[0]:
    return 1.0
  elif truth == predictions[1]:
    return 0.5
  elif truth == predictions[2]:
    return 0.3333
  else:
    return 0.0

truth = "C"
prediction = "C A B"

score = calculate_ap3(truth, prediction)
print("AP@3 Score is:", score)

AP@3 Score is: 1.0


8. If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

In [21]:
score = calculate_ap3("B", "D B E")
print("AP@3 Score is:", score)

AP@3 Score is: 0.5


9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [34]:
print(freq)
print(freq.idxmax())
most_frequent = freq.idxmax()
n_freq = freq.drop(index=most_frequent)
_2_most_frequent = n_freq.idxmax()
n_freq = n_freq.drop(index=_2_most_frequent)
_3_most_frequent = n_freq.idxmax()
prediction = f"{most_frequent} {_2_most_frequent} {_3_most_frequent}"
print("Top 3 frequent answers:", prediction)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
B
Top 3 frequent answers: B C A


In [40]:
num_questions = len(train_df)
score = 0
for index,row in train_df.iterrows():
  truth = row["answer"]
  score += calculate_ap3(truth, prediction)

print(f"MAP@3 for prediction: {prediction} is: {score/num_questions:.4f}")

MAP@3 for prediction: B C A is: 0.4212


10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [57]:
num_questions = len(train_df)
score = 0
for index, row in train_df.iterrows():
  prompt = row['prompt']
  options_data = [row['A'], row["B"], row["C"], row["D"], row["E"]]

  document = [prompt] + options_data
  vectors = tfidf_vectorizer.transform(document)
  similarities = cosine_similarity(vectors[:1], vectors[1:]).flatten()
  ranking = similarities.argsort()
  prediction = options[ranking[-1]] + " " + options[ranking[-2]] + " " + options[ranking[-3]]

  score += calculate_ap3(row["answer"], prediction)

print(f"Final Average MAP@3 score: {score/num_questions:.4f}")

[0.19968246 0.21072474 0.21017991 0.19930964 0.06774126]
[4 3 0 2 1]
B C A
Final Average MAP@3 score: 0.2408
